# Week 9 Mini Assignment — Build an Agent, Then Audit It (v3)

**ESP3201 · Codex v3 · AI Agents in Robotics Engineering**

This Colab progressively exposes the layers of an agentic system, using a **weather chatbot** as the
running example -- a domain everyone already understands, so the focus stays on the agent
machinery, not on decoding a robotics scenario:

```text
base LLM -> agent policy -> tools -> state + trace -> reflection
         -> component evals -> scenario eval -> planning -> agent team
```

You will call a **real, pinned model** (`nvidia/nemotron-3-nano-30b-a3b:free` via OpenRouter) and a **real,
free, keyless weather API** (Open-Meteo) -- so the tool results in this notebook are genuine
current data, not a simulation. The model never executes arbitrary code; a Python host validates
its JSON actions and exposes only a small, explicit set of tools.

> **Core rule:** model output proposes. Tool observations provide scoped evidence. A human owns the acceptance decision.

## 1. Learning goals and assessed outputs

By the end, you should be able to:

1. distinguish a base LLM response from an agent interacting with an environment;
2. explain the roles of policy, memory, tools, runtime, trace, evaluator, and human gate;
3. turn Python functions into bounded tools with explicit schemas and permissions;
4. compare reflection alone with revision grounded in external (real) evidence;
5. distinguish component-level checks from end-to-end scenario evidence;
6. validate a model-generated plan before execution; and
7. audit a planner/reviewer handoff without treating two models as independent proof.

**Submit:** selected live-run outputs, a component map, two trace analyses, an evidence ledger, one regression proposal, a bounded acceptance decision, and an AI-use disclosure. The final section provides the template.

## 2. Get a free OpenRouter API key

This notebook calls a real, **pinned** model: **`nvidia/nemotron-3-nano-30b-a3b:free`** on
OpenRouter -- a real, named model (not a rotating auto-router), so every student's run uses the
same model and results are comparable across the class. One HTTP request per turn. No GPU, no
local download.

1. Create a free account at https://openrouter.ai and generate a key at
   https://openrouter.ai/settings/keys.
2. **Recommended:** store it as a Colab Secret -- click the **key icon** in the left sidebar ->
   **Add new secret** -> name it exactly `OPENROUTER_API_KEY` -> paste your key -> toggle
   **notebook access** on. Secrets persist across sessions and are never written into the
   notebook file.
3. If you skip step 2, the next cell prompts you to paste the key directly (masked input, not
   echoed, not saved to disk, cleared when the runtime resets).

`RUN_LIVE=False` activates a scripted validation backend. It exists for repository testing and
instructor preview; students should use the real OpenRouter backend for submitted runs.

In [ ]:
RUN_LIVE = True

OPENROUTER_MODEL_ID = "nvidia/nemotron-3-nano-30b-a3b:free"   # pinned -- see section 2
MAX_NEW_TOKENS = 800
TEMPERATURE = 0.1

print({
    "run_live": RUN_LIVE,
    "openrouter_model": OPENROUTER_MODEL_ID,
})

In [ ]:
import os
from getpass import getpass

def get_openrouter_key() -> str:
    'Resolve OPENROUTER_API_KEY: Colab Secrets -> environment -> interactive prompt.'
    try:
        from google.colab import userdata
        key = userdata.get("OPENROUTER_API_KEY")
        if key:
            print("Using OPENROUTER_API_KEY from Colab Secrets.")
            return key
    except Exception:
        pass

    key = os.environ.get("OPENROUTER_API_KEY", "")
    if key:
        print("Using OPENROUTER_API_KEY from the environment.")
        return key

    print("No OPENROUTER_API_KEY found in Colab Secrets or the environment.")
    key = getpass("Paste your OpenRouter API key (input hidden, not saved to disk): ").strip()
    if not key:
        raise RuntimeError(
            "An OpenRouter API key is required. Get a free key at https://openrouter.ai/settings/keys"
        )
    return key

if RUN_LIVE:
    OPENROUTER_API_KEY = get_openrouter_key()
    os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
else:
    OPENROUTER_API_KEY = ""
    print("RUN_LIVE is False: skipping API key resolution (scripted backend only).")

In [ ]:
from dataclasses import dataclass
import json
import os
import re
from typing import Any

@dataclass
class ModelReply:
    text: str
    model: str

class OpenRouterBackend:
    'OpenAI-compatible HTTP call to OpenRouter.'

    def __init__(self, model_id: str):
        self.model_id = model_id
        self.api_key = os.environ.get("OPENROUTER_API_KEY", "")
        if not self.api_key:
            raise RuntimeError(
                "No OPENROUTER_API_KEY set. Re-run the API-key cell in section 2 above."
            )

    def chat(self, messages, *, temperature=TEMPERATURE, max_new_tokens=MAX_NEW_TOKENS):
        import requests

        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {self.api_key}",
                "Content-Type": "application/json",
            },
            json={
                "model": self.model_id,
                "messages": messages,
                "temperature": temperature,
                "max_tokens": max_new_tokens,
            },
            timeout=120,
        )
        response.raise_for_status()
        data = response.json()
        message = data["choices"][0].get("message", {})
        # gpt-oss (trained in OpenAI's "Harmony" response format) and other reasoning
        # models are documented to sometimes return content: null on some providers --
        # e.g. when a turn is entirely consumed by reasoning tokens. Treat that as an
        # empty reply rather than crashing: extract_json_object's "No JSON object found"
        # path already turns an empty/unparseable reply into a recorded ERROR trace
        # event instead of stopping the notebook (see the agent-runtime cell).
        text = (message.get("content") or "").strip()
        return ModelReply(text=text, model=data.get("model", self.model_id))

class ScriptedBackend:
    'Deterministic infrastructure test double; not the student live path.'

    model_id = "scripted-validation"

    def chat(self, messages, **_):
        prompt = messages[-1]["content"]
        lower = prompt.lower()
        if "direct answer" in lower:
            text = (
                "It's sunny and about 30C in Singapore right now, with a light breeze."
            )
        elif "critic with no new evidence" in lower:
            text = (
                "The phrasing is clear and the number is plausible for the season. "
                "I cannot verify the actual temperature or conditions from the candidate alone."
            )
        elif "revise using scenario feedback" in lower:
            text = (
                "Candidate v2: using the retrieved reading, Singapore is currently colder/warmer "
                "than my first guess -- see the tool observation for the exact figure. This "
                "revision is grounded in a real API call, not a repeated guess."
            )
        elif "return a plan json" in lower:
            text = json.dumps({
                "plan": [
                    {"tool": "get_weather", "args": {"location": "Berlin"}},
                    {"tool": "remember_last_location", "args": {"location": "Berlin"}},
                ],
                "rationale": "look up the real weather, then save the city as the session default",
            })
        elif "fact-check reviewer" in lower:
            text = (
                "Faithful to the retrieved data for the temperature figure. Unverified: whether "
                "the remembered location will still be used correctly in a later session."
            )
        elif "return exactly one json object" in lower:
            trace = []
            marker = "TRACE_JSON:\n"
            if marker in prompt:
                try:
                    trace = json.loads(prompt.split(marker, 1)[1])
                except Exception:
                    trace = []
            actions = [e.get("tool") for e in trace if e.get("kind") == "ACTION"]
            has_weather = "get_weather" in actions
            has_remember = "remember_last_location" in prompt.split("TRACE_JSON:", 1)[0]
            if not has_weather:
                obj = {"thought": "look up the real current weather first",
                       "action": {"tool": "get_weather", "args": {"location": "Tokyo"}}}
            elif "remember_last_location" not in actions and has_remember:
                obj = {"thought": "save this city as the session default",
                       "action": {"tool": "remember_last_location", "args": {"location": "Tokyo"}}}
            elif not has_remember:
                obj = {"thought": "the remember tool is unavailable with this tool set",
                       "final": "I looked up the weather, but I cannot remember a default city -- that tool isn't available to me."}
            else:
                obj = {"thought": "bound the claim to what was actually retrieved",
                       "final": "Looked up Tokyo's current weather and saved it as the session default."}
            text = json.dumps(obj)
        else:
            text = "A model response was requested."
        return ModelReply(text=text, model=self.model_id)

def build_backend():
    if not RUN_LIVE:
        return ScriptedBackend()
    return OpenRouterBackend(OPENROUTER_MODEL_ID)

In [ ]:
backend = build_backend()
print("Backend ready:", type(backend).__name__)

In [ ]:
# Diagram helper -- run once. Draws the boxes-and-arrows figures used throughout this notebook.
# Blue = new this layer; gray = carried over from an earlier layer; green = evidence/observation;
# red = a claim or path that is NOT yet backed by evidence.
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

_STATE_STYLE = {
    "new":      dict(facecolor="#e9f1fd", edgecolor="#2563eb", linewidth=2.2, fontweight="bold", fontcolor="#1d4ed8"),
    "carried":  dict(facecolor="#f1f5f9", edgecolor="#94a3b8", linewidth=1.3, fontweight="normal", fontcolor="#475569"),
    "evidence": dict(facecolor="#e6f5ef", edgecolor="#047857", linewidth=2.0, fontweight="bold", fontcolor="#036c4e"),
    "risk":     dict(facecolor="#fdecea", edgecolor="#e74c3c", linewidth=1.8, fontweight="bold", fontcolor="#c0392b"),
}
_BOX_W, _BOX_H, _GAP = 2.15, 0.92, 0.55
_ROW_H = 1.35
_LOOP_LANE = 1.05

def _draw_row(ax, y, stages, arrows=True):
    x = 0.0
    centers = []
    for st in stages:
        style = _STATE_STYLE[st.get("state", "carried")]
        box = FancyBboxPatch((x, y), _BOX_W, _BOX_H, boxstyle="round,pad=0.02,rounding_size=0.09",
                              linewidth=style["linewidth"], edgecolor=style["edgecolor"],
                              facecolor=style["facecolor"], zorder=2)
        ax.add_patch(box)
        cx, cy = x + _BOX_W / 2, y + _BOX_H / 2
        sub = st.get("sub")
        ax.text(cx, cy + (0.15 if sub else 0), st["label"], ha="center", va="center",
                fontsize=10.3, fontweight=style["fontweight"], color=style["fontcolor"], zorder=3)
        if sub:
            ax.text(cx, cy - 0.22, sub, ha="center", va="center", fontsize=8.4,
                     color=style["fontcolor"], zorder=3)
        centers.append((cx, cy))
        x += _BOX_W + _GAP
    if arrows:
        for (x1, y1), (x2, y2) in zip(centers, centers[1:]):
            ax.annotate("", xy=(x2 - _BOX_W / 2 - 0.04, y2), xytext=(x1 + _BOX_W / 2 + 0.04, y1),
                        arrowprops=dict(arrowstyle="-|>", color="#64748b", lw=1.5), zorder=1)
    return x - _GAP

def show_layer_diagram(title, rows, note=None, figsize=None):
    # rows: list of {"label": row caption, "stages": [{"label","sub","state"}], "arrows": bool,
    # "loop": bool, "loop_label": str} -- draws left-to-right chains stacked top to bottom.
    #
    # Renders exactly one figure. (An earlier version both called plt.show() AND returned the
    # Figure -- when a cell's only statement is show_layer_diagram(...), Jupyter/Colab renders
    # the returned Figure a second time as the cell's "result", producing two copies of the same
    # picture. This version shows the figure and closes it, returning nothing.)
    heights = [_ROW_H + (_LOOP_LANE if r.get("loop") else 0) for r in rows]
    total_h = sum(heights)
    has_labels = any(r.get("label") for r in rows)
    fig_h = 0.55 + total_h + (0.35 if note else 0)
    fig, ax = plt.subplots(figsize=figsize or (11.8, fig_h))
    ax.axis("off")
    max_x = 0
    y_cursor = total_h
    for row in rows:
        y_cursor -= _ROW_H
        y = y_cursor
        rx = _draw_row(ax, y, row["stages"], arrows=row.get("arrows", True))
        max_x = max(max_x, rx)
        if row.get("label"):
            ax.text(-0.2, y + _BOX_H / 2, row["label"], ha="right", va="center",
                     fontsize=9.3, fontweight="bold", color="#334155")
        if row.get("loop"):
            stages = row["stages"]
            last_x = (len(stages) - 1) * (_BOX_W + _GAP) + _BOX_W / 2
            first_x = _BOX_W / 2
            arc_y = y - _LOOP_LANE * 0.42
            ax.annotate("", xy=(first_x, y - 0.05), xytext=(last_x, y - 0.05),
                        arrowprops=dict(arrowstyle="-|>", color="#94a3b8", lw=1.4,
                                         linestyle=(0, (5, 3)), connectionstyle="arc3,rad=-0.28"))
            ax.text((first_x + last_x) / 2, arc_y - 0.06,
                     row.get("loop_label", "repeats until FINAL"),
                     ha="center", va="center", fontsize=8.3, color="#64748b", style="italic")
            y_cursor -= _LOOP_LANE
    ax.set_xlim(-3.0 if has_labels else -0.3, max_x + 0.4)
    ax.set_ylim(-0.35, total_h + 0.35)
    ax.set_title(title, fontsize=12.5, fontweight="bold", color="#1a1a2e", loc="left", pad=8)
    if note:
        fig.text(0.02, 0.01, note, fontsize=8.8, color="#64748b", style="italic")
    plt.tight_layout()
    plt.show()
    plt.close(fig)

DIAGRAMS = {
    "d1_base_llm": dict(
        title="Layer 1 -- A Base LLM Call",
        rows=[{"stages": [
            {"label": "TASK", "sub": '"weather in Singapore?"', "state": "carried"},
            {"label": "MODEL", "sub": "nvidia/nemotron-3-nano-30b-a3b:free", "state": "new"},
            {"label": "TEXT RESPONSE", "sub": "no observation, no trace", "state": "risk"},
        ]}],
        note="No environment, no tools, no loop -- a fluent-sounding number is not yet evidence of the real temperature.",
    ),
    "d2_environment": dict(
        title="Layer 2 -- Environment and State (ChatSession)",
        rows=[{"arrows": False, "stages": [
            {"label": "conversation_turns", "sub": "how many turns so far", "state": "carried"},
            {"label": "last_location", "sub": "remembered default city", "state": "carried"},
            {"label": "last_weather", "sub": "cached last lookup", "state": "carried"},
        ]}],
        note="One small object, three fields -- this is the chatbot's memory. Not yet connected to a model in this section.",
    ),
    "d3_tools": dict(
        title="Layer 3 -- Turn Functions into Bounded Tools",
        rows=[{"stages": [
            {"label": "MODEL", "sub": "proposes tool + args", "state": "new"},
            {"label": "HOST", "sub": "ToolRegistry.execute()", "state": "new"},
            {"label": "VALIDATE", "sub": "name + schema check", "state": "new"},
            {"label": "Open-Meteo API", "sub": "real, free, keyless", "state": "evidence"},
        ]}],
        note="The model never calls the weather API directly -- it only requests a tool by name; the host validates and makes the real HTTP call.",
    ),
    "d4_agent_loop": dict(
        title="Layer 4 -- Base Agent Loop, Memory, and Trace",
        rows=[
            {
                "label": "loop", "loop": True,
                "loop_label": "observation feeds the next MODEL turn -- repeats until FINAL or max_steps",
                "stages": [
                    {"label": "TASK + TRACE", "sub": "growing JSON log", "state": "carried"},
                    {"label": "MODEL", "sub": "thought + action json", "state": "new"},
                    {"label": "HOST", "sub": "validate + execute tool", "state": "new"},
                    {"label": "OBSERVATION", "sub": "appended to trace", "state": "evidence"},
                ],
            },
            {"label": "exit", "stages": [
                {"label": "MODEL", "sub": "thought + final", "state": "new"},
                {"label": "AgentRun", "sub": "final, trace, models", "state": "evidence"},
            ]},
        ],
        note="A malformed action (not a {tool,args} object) or an empty model reply becomes an ERROR trace event here -- not a crash.",
    ),
    "d5_reflection": dict(
        title="Layer 5 -- Reflection versus External (Real) Evidence",
        rows=[
            {"label": "reflection\nonly", "stages": [
                {"label": "CANDIDATE", "sub": "direct guess", "state": "carried"},
                {"label": "CRITIC MODEL", "sub": "no new evidence", "state": "carried"},
                {"label": "REVISED TEXT", "sub": "same guess, better prose", "state": "risk"},
            ]},
            {"label": "evidence-\ngrounded", "stages": [
                {"label": "CANDIDATE", "sub": "direct guess", "state": "carried"},
                {"label": "Open-Meteo API", "sub": "real current reading", "state": "evidence"},
                {"label": "MODEL", "sub": "revise using the reading", "state": "new"},
                {"label": "GROUNDED REVISION", "sub": "bounded by real data", "state": "evidence"},
            ]},
        ],
        note="Only the row with a real API call changes what is known about the actual weather.",
    ),
    "d6_evals": dict(
        title="Layer 6 -- Component Evals versus End-to-End Eval",
        rows=[
            {"label": "component\n(cheap)", "arrows": False, "stages": [
                {"label": "tool_contracts", "sub": "schema check -- PASS", "state": "evidence"},
                {"label": "weather_response_shape", "sub": "has numeric fields -- PASS", "state": "evidence"},
            ]},
            {"label": "end-to-end\n(expensive)", "arrows": False, "stages": [
                {"label": "answer matches tool data?", "sub": "compare stated vs retrieved number", "state": "risk"},
            ]},
        ],
        note="Both component checks can pass while the model still states a different number in its final prose than the tool actually returned.",
    ),
    "d7_planning": dict(
        title="Layer 7 -- Planning Without Arbitrary Code Execution",
        rows=[{"stages": [
            {"label": "MODEL", "sub": "plan json: [{tool,args}, ...]", "state": "new"},
            {"label": "HOST", "sub": "validate_plan()", "state": "new"},
            {"label": "EXECUTE STEPS", "sub": "sequential, allowlisted", "state": "evidence"},
            {"label": "EXECUTION TRACE", "sub": "per-step observation", "state": "evidence"},
        ]}],
        note="No exec() on model output -- every step's tool name is checked against the registry before it runs.",
    ),
    "d8_multiagent": dict(
        title="Layer 8 -- Planner and Fact-Check Reviewer",
        rows=[{"stages": [
            {"label": "PLANNER MODEL", "sub": "plan + execution trace", "state": "new"},
            {"label": "HANDOFF RECORD", "sub": "model, trace IDs, tool data", "state": "carried"},
            {"label": "REVIEWER MODEL", "sub": "sees only the handoff", "state": "new"},
            {"label": "VERDICT", "sub": "faithful / unfaithful + what's unverified", "state": "evidence"},
        ]}],
        note="The reviewer is only as independent as the evidence in the handoff -- two models agreeing is not proof the final answer is faithful to the data.",
    ),
}
print('Diagram helpers ready:', list(DIAGRAMS.keys()))

# Shared colored, labeled block display -- used by print_trace() below and by the
# reflection flow (Layer 5) so every step-by-step trace in this notebook reads the
# same way: one color per role, a bold uppercase label, wrapped (not truncated) text.
from html import escape as _esc
from IPython.display import HTML, display as _display

_BLOCK_COLORS = {
    "model":       ("#2563eb", "#e9f1fd"),  # blue -- a claim / thought / candidate
    "action":      ("#d97706", "#fdf3e3"),  # amber -- a tool call request
    "observation": ("#047857", "#e6f5ef"),  # green -- real evidence
    "error":       ("#e74c3c", "#fdecea"),  # red -- malformed / failed
    "final":       ("#036c4e", "#d1fae5"),  # dark green, heavier -- the final output
    "note":        ("#94a3b8", "#f1f5f9"),  # gray -- no new evidence / commentary
}

def show_section_header(text):
    _display(HTML(
        f'<div style="font-weight:800; font-size:14px; letter-spacing:.03em; '
        f'margin:14px 0 4px; padding:8px 12px; border-radius:6px; '
        f'background:#dbeafe; color:#1a1a2e;">{_esc(text)}</div>'
    ))

def show_block(label, role, text):
    edge, bg = _BLOCK_COLORS.get(role, ("#94a3b8", "#f1f5f9"))
    _display(HTML(
        f'<div style="border-left:4px solid {edge}; background:{bg}; border-radius:6px; '
        f'padding:8px 12px; margin:4px 0; font-family:system-ui,-apple-system,sans-serif;">'
        f'<div style="font-weight:700; font-size:11px; letter-spacing:.05em; text-transform:uppercase; '
        f'color:{edge}; margin-bottom:4px;">{_esc(label)}</div>'
        f'<div style="font-size:13.5px; color:#1a1a2e; white-space:pre-wrap; '
        f'font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;">{_esc(text)}</div>'
        f'</div>'
    ))

print("Shared block display ready: show_block, show_section_header")

## 3. Layer 1 -- A base LLM call

First, ask the model directly. It receives a text question but has no environment, tools, or
observations -- it cannot actually look anything up.

Record what makes the answer sound plausible -- and which factual claims cannot be checked from
the response alone.

In [ ]:
show_layer_diagram(**DIAGRAMS["d1_base_llm"])

In [ ]:
TASK = "What is the weather like in Singapore right now?"
direct_reply = backend.chat([
    {"role": "system", "content": "You are a weather assistant."},
    {"role": "user", "content": TASK},
])
print("MODEL:", direct_reply.model)
print(direct_reply.text)

### Checkpoint A -- response versus evidence

In your submission, quote one claim from the direct answer and complete:

| Claim | Why it is plausible | Missing observation | Smallest adequate verification layer |
| --- | --- | --- | --- |
| | | | |

This is an LLM application, not yet an agent: it has not acted on or observed anything real.

## 4. Layer 2 -- Environment and state

An agent needs something to interact with, and something to remember across turns. `ChatSession`
is deliberately small: how many turns have happened, which city (if any) the user asked to be
remembered as the default, and the last weather reading fetched.

In [ ]:
show_layer_diagram(**DIAGRAMS["d2_environment"])

In [ ]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class ChatSession:
    conversation_turns: int = 0
    last_location: Optional[str] = None
    last_weather: Optional[dict] = None

    def reset(self):
        self.conversation_turns = 0
        self.last_location = None
        self.last_weather = None
        return self.snapshot()

    def snapshot(self):
        return {
            "conversation_turns": self.conversation_turns,
            "last_location": self.last_location,
            "last_weather": self.last_weather,
        }

    def remember(self, location, weather=None):
        before = self.last_location
        self.last_location = location
        if weather is not None:
            self.last_weather = weather
        self.conversation_turns += 1
        return {"event": "remembered", "before": before, "after": location}

session = ChatSession()
session.snapshot()

In [ ]:
print("Initial session state:")
print(json.dumps(session.snapshot(), indent=2))

## 5. Layer 3 -- Turn functions into bounded tools

The model does not call the weather API itself. It requests a tool by name and arguments; the
host validates the request, calls the function -- which makes a **real HTTP call** to Open-Meteo
(free, keyless) -- and returns an observation.

We begin with a **read-only tool set**. The missing "remember" tool creates an intentional
capability boundary.

In [ ]:
show_layer_diagram(**DIAGRAMS["d3_tools"])

In [ ]:
import requests
from datetime import datetime
from zoneinfo import ZoneInfo
from dataclasses import dataclass
from typing import Callable

def geocode_location(location: str) -> dict:
    """Turn a place name into coordinates + timezone via Open-Meteo's free geocoding API."""
    resp = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                        params={"name": location, "count": 1}, timeout=10)
    resp.raise_for_status()
    results = resp.json().get("results")
    if not results:
        raise ValueError(f"Could not find a location matching {location!r}")
    r = results[0]
    return {"name": r["name"], "country": r.get("country_code", ""),
            "latitude": r["latitude"], "longitude": r["longitude"], "timezone": r["timezone"]}

def get_weather(location: str) -> dict:
    """Look up the current real-world weather for a place name (Open-Meteo, free, no key)."""
    geo = geocode_location(location)
    resp = requests.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude": geo["latitude"], "longitude": geo["longitude"],
        "current": "temperature_2m,windspeed_10m", "timezone": "auto",
    }, timeout=10)
    resp.raise_for_status()
    cur = resp.json()["current"]
    result = {
        "location": geo["name"], "country": geo["country"],
        "temperature_c": cur["temperature_2m"], "windspeed_kmh": cur["windspeed_10m"],
        "observed_at": cur["time"],
    }
    session.last_weather = result
    return result

def get_local_time(location: str) -> dict:
    """Look up the current real-world local time for a place name (no weather)."""
    geo = geocode_location(location)
    now = datetime.now(ZoneInfo(geo["timezone"]))
    return {"location": geo["name"], "timezone": geo["timezone"],
            "local_time": now.strftime("%Y-%m-%d %H:%M:%S")}

def remember_last_location(location: str) -> dict:
    """Save a location as this session's remembered default city."""
    return session.remember(location)

@dataclass
class ToolSpec:
    name: str
    description: str
    parameters: dict
    effect: str
    function: Callable[..., Any]

    def public_schema(self):
        return {
            "name": self.name,
            "description": self.description,
            "parameters": self.parameters,
            "effect": self.effect,
        }

class ToolRegistry:
    def __init__(self, tools):
        self.tools = {tool.name: tool for tool in tools}

    def schemas(self):
        return [tool.public_schema() for tool in self.tools.values()]

    def execute(self, name, args):
        if name not in self.tools:
            raise PermissionError(f"Tool not allowed: {name}")
        spec = self.tools[name]
        args = args or {}
        allowed = set(spec.parameters.get("properties", {}))
        required = set(spec.parameters.get("required", []))
        if set(args) - allowed:
            raise ValueError(f"Unexpected arguments for {name}: {set(args) - allowed}")
        if required - set(args):
            raise ValueError(f"Missing arguments for {name}: {required - set(args)}")
        return spec.function(**args)

LOCATION_ARGS = {
    "type": "object",
    "properties": {"location": {"type": "string", "description": "A city name, e.g. 'Tokyo'."}},
    "required": ["location"],
    "additionalProperties": False,
}
EMPTY_ARGS = {"type": "object", "properties": {}, "required": [], "additionalProperties": False}

read_status = ToolSpec(
    "read_status", "Read this chat session's remembered state (no API call).",
    EMPTY_ARGS, "read-only", session.snapshot,
)
get_weather_tool = ToolSpec(
    "get_weather", "Look up the current real-world weather for a place name.",
    LOCATION_ARGS, "read-only; calls a live external API", get_weather,
)
get_local_time_tool = ToolSpec(
    "get_local_time", "Look up the current real-world local time for a place name.",
    LOCATION_ARGS, "read-only; calls a live external API", get_local_time,
)
remember_location_tool = ToolSpec(
    "remember_last_location", "Save a location as the session's remembered default city.",
    LOCATION_ARGS, "state-changing; session only", remember_last_location,
)

read_only_registry = ToolRegistry([read_status, get_weather_tool, get_local_time_tool])
full_registry = ToolRegistry([read_status, get_weather_tool, get_local_time_tool, remember_location_tool])

In [ ]:
print("READ-ONLY ACTION SPACE")
print(json.dumps(read_only_registry.schemas(), indent=2))
print("\nFULL ACTION SPACE")
print(json.dumps(full_registry.schemas(), indent=2))

**Raw tool calls, no agent involved yet:**

In [ ]:
# Before wrapping these in an agent loop, call them directly so you can see
# exactly what goes in and what real data comes back.
print("geocode_location('Singapore') ->")
print(json.dumps(geocode_location("Singapore"), indent=2))

print("\nget_weather('Singapore') ->")
print(json.dumps(get_weather("Singapore"), indent=2))

print("\nget_local_time('Singapore') ->")
print(json.dumps(get_local_time("Singapore"), indent=2))

## 6. Layer 4 -- Base agent loop, memory, and trace

The runtime below adds what the base LLM lacked:

- an action protocol;
- an allowed tool registry;
- state carried through observations;
- a bounded loop and stop condition; and
- a trace that separates model text, action request, host execution, and observation.

The model must return one JSON object per step:

```json
{"thought":"...", "action":{"tool":"get_weather", "args":{"location":"Tokyo"}}}
```

or a bounded final response:

```json
{"thought":"...", "final":"..."}
```

In [ ]:
show_layer_diagram(**DIAGRAMS["d4_agent_loop"])

In [ ]:
def extract_json_object(text: str) -> dict:
    'Parse a JSON object even when a small model adds a code fence.'
    cleaned = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start < 0 or end < start:
        raise ValueError("No JSON object found")
    return json.loads(cleaned[start:end + 1])

@dataclass
class AgentRun:
    final: str
    trace: list[dict]
    models: list[str]

class AgentRuntime:
    def __init__(self, model_backend, registry, *, max_steps=7):
        self.backend = model_backend
        self.registry = registry
        self.max_steps = max_steps

    def run(self, task: str):
        trace, models = [], []
        for step in range(self.max_steps):
            prompt = f'''
TASK:
{task}

AVAILABLE_TOOLS:
{json.dumps(self.registry.schemas(), indent=2)}

Return exactly one JSON object. Choose one allowed tool action, or return final only
when the trace contains enough evidence. Never invent a weather reading or a location
you have not actually looked up.

Action must be an object: {{"tool": "<name>", "args": {{}}}}. A bare tool name is invalid.

Once you have enough evidence, respond with a JSON object containing a "final" key whose
value is your own one-sentence answer written in plain words -- never a template or placeholder.
Do not just restate the retrieved data as a bare JSON object -- wrap your answer in "final".

TRACE_JSON:
{json.dumps(trace)}
'''.strip()
            reply = self.backend.chat([
                {"role": "system", "content": "You are a careful weather-chatbot agent using a host-controlled tool loop."},
                {"role": "user", "content": prompt},
            ])
            models.append(reply.model)
            event_id = f"m{step + 1:02d}"
            trace.append({"id": event_id, "kind": "MODEL", "model": reply.model, "content": reply.text})
            try:
                decision = extract_json_object(reply.text)
            except Exception as exc:
                trace.append({"id": f"e{step + 1:02d}", "kind": "ERROR", "content": f"invalid JSON: {exc}"})
                continue

            if "final" in decision:
                final_text = str(decision["final"]).strip()
                # A model under pressure sometimes echoes the prompt's own instruction text
                # verbatim (e.g. a literal "<a short answer to the user>") instead of writing a
                # real answer -- valid JSON, present key, but not actually evidence of anything.
                # This is exactly the "fluent output is not evidence" failure the course teaches:
                # a template echo would otherwise sail through silently as a normal final answer.
                if not final_text or (final_text.startswith("<") and final_text.endswith(">")):
                    trace.append({
                        "id": f"e{step + 1:02d}",
                        "kind": "ERROR",
                        "content": f"'final' looked like an unfilled template placeholder, not a real answer: {final_text!r}",
                    })
                    continue
                trace.append({"id": f"f{step + 1:02d}", "kind": "FINAL", "content": final_text})
                return AgentRun(final_text, trace, models)

            # A small model frequently flattens this to {"action": "tool_name"} instead of
            # {"action": {"tool": "tool_name", "args": {}}}. That is malformed-but-parseable
            # output, not a crash: record it as trace evidence and let the loop continue,
            # exactly like the invalid-JSON branch above.
            action = decision.get("action")
            if not isinstance(action, dict) and isinstance(decision.get("tool"), str):
                # Small models commonly flatten this to {"tool":..., "args":...} at the top
                # level instead of nesting under "action" (observed live with several free
                # OpenRouter models). That is an unambiguous, equivalent shape, not a
                # malformed one; recognizing it here avoids burning the whole step budget
                # on ERROR events for output the model expressed clearly.
                action = {"tool": decision.get("tool"), "args": decision.get("args", {})}
            if not isinstance(action, dict):
                # Once real evidence is already in the trace, a model sometimes just restates
                # the collected data as a bare JSON object -- e.g. {"weather": {...}} -- instead
                # of wrapping it in {"final": "..."}. Treat that as an IMPLICIT final answer
                # rather than erroring every remaining step with zero progress. Guarded on at
                # least one real OBSERVATION already existing: an evidence-free reply shaped
                # like this on turn one is a confused first guess, not a final answer, and
                # accepting it as one would be exactly the "confident but unverified" mistake
                # this course's evidence-vs-claim throughline warns against -- so that case
                # still becomes an ERROR, unchanged.
                has_observation = any(e.get("kind") == "OBSERVATION" for e in trace)
                if has_observation and decision:
                    implicit_final = json.dumps(decision)
                    trace.append({
                        "id": f"f{step + 1:02d}",
                        "kind": "FINAL",
                        "content": implicit_final,
                        "note": "no 'final' key in the reply; treated as an implicit final answer "
                                "because the trace already contains a real observation",
                    })
                    return AgentRun(implicit_final, trace, models)
                trace.append({
                    "id": f"e{step + 1:02d}",
                    "kind": "ERROR",
                    "content": (
                        f"'action' was not an object (got {type(action).__name__}: {action!r}); "
                        'expected {"tool": ..., "args": {...}}'
                    ),
                })
                continue

            tool_name, args = action.get("tool"), action.get("args", {})
            trace.append({"id": f"a{step + 1:02d}", "kind": "ACTION", "tool": tool_name, "args": args})
            try:
                observation = self.registry.execute(tool_name, args)
                trace.append({"id": f"o{step + 1:02d}", "kind": "OBSERVATION", "tool": tool_name, "content": observation})
            except Exception as exc:
                trace.append({"id": f"o{step + 1:02d}", "kind": "OBSERVATION", "tool": tool_name, "content": {"error": str(exc)}})

        final = "Stopped at the step limit; no accepted conclusion."
        trace.append({"id": "limit", "kind": "FINAL", "content": final})
        return AgentRun(final, trace, models)

def print_trace(run: AgentRun):
    role_by_kind = {"MODEL": "model", "ACTION": "action", "OBSERVATION": "observation",
                     "ERROR": "error", "FINAL": "final"}
    for event in run.trace:
        kind = event.get("kind", "")
        role = role_by_kind.get(kind, "model")
        tool = event.get("tool")
        label = f"{event.get('id')} -- {kind}" + (f" -> {tool}" if tool else "")
        if kind == "FINAL":
            label = f"{event.get('id')} -- FINAL OUTPUT"
        content = event.get("content", event.get("args", ""))
        text = content if isinstance(content, str) else json.dumps(content, indent=2, default=str)
        show_block(label, role, text)
    print("MODELS USED:", sorted(set(run.models)))

### 6.1 Base agent with a missing capability

Run the real model with the read-only tools only, across **two separate conversations**:

- **Conversation A** asks it to look up the weather *and* remember a default city for next time.
- **Conversation B** -- a later turn -- asks it to look up the weather in "my current city",
  with nothing said about which city that is.

The read-only agent has no `remember_last_location` tool, so nothing is ever actually saved.
Conversation A alone can look fine (it can still answer the weather half); the missing capability
only becomes visible in Conversation B, when the agent has to admit it does not know "my current
city" -- or fails to.

Look for three possibilities in Conversation B -- all are instructive:

- it correctly admits the capability gap (no known current city);
- it requests a tool that the host rejects; or
- it overclaims despite the missing tool (e.g. guesses a city, or says "as I noted before" when
  nothing was ever recorded).


In [ ]:
session.reset()
restricted_agent = AgentRuntime(backend, read_only_registry, max_steps=5)

restricted_run_remember = restricted_agent.run(
    "Look up the current weather in Tokyo, and remember Tokyo as my default city for next time. "
    "State only what the evidence supports."
)
print("=== Conversation A: asks the agent to remember a default city ===")
print_trace(restricted_run_remember)

restricted_run_recall = restricted_agent.run(
    "Look up the current weather in my current city. "
    "State only what the evidence supports."
)
print("\n=== Conversation B: a later turn that depends on that memory existing ===")
print_trace(restricted_run_recall)


### Checkpoint B -- capability is not permission to claim success

Cite the exact trace IDs for:

1. Conversation A's first external observation (the real weather reading);
2. any invalid or disallowed tool request in either conversation; and
3. Conversation B's final claim.

Did Conversation B's final claim stay within the evidence? Since nothing was ever actually saved,
what *should* an honest answer to "my current city" look like -- and did the model give one?


### 6.2 Add the state-changing tool

The next run adds `remember_last_location` and repeats the same two-conversation structure. This
may let the agent complete the full task -- Conversation B should now be answerable, because
Conversation A had a real tool available to record the default city. The host still validates and
records every action.


In [ ]:
session.reset()
full_agent = AgentRuntime(backend, full_registry, max_steps=8)

full_run_remember = full_agent.run(
    "Look up the current weather in Tokyo, and remember Tokyo as my default city for next time. "
    "State only what the evidence supports."
)
print("=== Conversation A: asks the agent to remember a default city ===")
print_trace(full_run_remember)

full_run_recall = full_agent.run(
    "Look up the current weather in my current city. "
    "State only what the evidence supports."
)
print("\n=== Conversation B: a later turn that depends on that memory existing ===")
print_trace(full_run_recall)


In [ ]:
def trace_table(run):
    rows = []
    for event in run.trace:
        content = event.get("content", event.get("args", ""))
        summary = content if isinstance(content, str) else json.dumps(content, default=str)
        rows.append({
            "id": event.get("id"),
            "kind": event.get("kind"),
            "tool": event.get("tool", ""),
            "summary": summary[:120],
        })
    return rows

print("Conversation B -- with memory (should now resolve \"my current city\"):")
for row in trace_table(full_run_recall):
    print(row)


## 7. Layer 5 -- Reflection versus external (real) feedback

Adapt the reflection pattern from the reference notebooks:

1. ask a critic to review a candidate **without** new observations;
2. call the real `get_weather` tool to obtain external evidence; and
3. ask for a revision using that real reading.

Reflection can improve a candidate's wording. Only a real observation changes what is known about
the actual weather.

In [ ]:
show_layer_diagram(**DIAGRAMS["d5_reflection"])

In [ ]:
candidate = direct_reply.text

reflection_only = backend.chat([
    {"role": "system", "content": "You are a critic with no new evidence."},
    {"role": "user", "content": f"CRITIC WITH NO NEW EVIDENCE. Review this candidate:\n{candidate}"},
])

reflection_only_revision = backend.chat([
    {"role": "system", "content": (
        "Revise a candidate using only the critique below -- you have no new evidence. "
        "If the critique shows the candidate's core claim is unsupported, your revision must say so "
        "honestly (e.g. state plainly that you cannot verify current conditions) rather than restate "
        "the unsupported claim in different words. Respond with ONLY the revised answer text itself -- "
        "no reasoning, no preamble, no meta-commentary about the task."
    )},
    {"role": "user", "content": (
        "REVISE USING ONLY THIS CRITIQUE -- NO NEW EVIDENCE IS AVAILABLE.\n"
        f"Candidate:\n{candidate}\n\n"
        f"Critique:\n{reflection_only.text}"
    )},
])

session.reset()
real_reading = get_weather("Singapore")
grounded_revision = backend.chat([
    {"role": "system", "content": (
        "Revise a candidate using a supplied real API reading. Do not broaden its scope. "
        "Respond with ONLY the revised answer text itself -- no reasoning, no preamble, no "
        "meta-commentary about the task."
    )},
    {"role": "user", "content": (
        "REVISE USING SCENARIO FEEDBACK.\n"
        f"Candidate:\n{candidate}\n\n"
        f"REAL API READING:\n{json.dumps(real_reading, indent=2)}"
    )},
])

show_section_header("REFLECTION-ONLY FLOW -- no new evidence enters at any step")
show_block("1. Candidate -- direct answer, no evidence", "model", candidate)
show_block(f"2. Critic model ({reflection_only.model}) -- reviews the candidate alone", "note", reflection_only.text)
show_block(f"3. Reflection-only revision ({reflection_only_revision.model}) -- revised using only the critique, still no evidence", "model", reflection_only_revision.text)
show_block("4. FINAL OUTPUT -- what the user would actually see", "final", reflection_only_revision.text)

show_section_header("EVIDENCE-GROUNDED FLOW -- a real tool call enters between candidate and revision")
show_block("1. Candidate -- direct answer, no evidence", "model", candidate)
show_block("2. Real tool observation -- get_weather(\"Singapore\")", "observation", json.dumps(real_reading, indent=2))
show_block(f"3. Grounded revision ({grounded_revision.model}) -- rewritten using the reading above", "model", grounded_revision.text)
show_block("4. FINAL OUTPUT -- what the user would actually see", "final", grounded_revision.text)

### Checkpoint C -- what introduced new information?

Compare the two FINAL OUTPUTs (reflection-only vs. evidence-grounded) and the real API reading.
Identify:

- one useful suggestion either critic could make;
- one statement supported only after the real `get_weather` call ran;
- one claim in the direct answer that turned out to be wrong (or right by luck) once checked
  against the real reading; and
- whether the reflection-only FINAL OUTPUT honestly admitted it could not verify current
  conditions, or quietly restated an unsupported claim instead.


## 8. Layer 6 -- Component evals versus end-to-end eval

A component evaluator is cheap and diagnostic. An end-to-end scenario asks whether the pieces
compose into the behaviour we actually care about: **did the model's final answer accurately
state what the tool really returned?**

Passing both component checks below does **not** prove the final answer is faithful: neither one
looks at the model's own prose.

**A weakness worth sitting with.** The end-to-end check below is a regex: it looks for a number
in temperature context and calls it a pass if one is close enough to the real reading. That is
still a heuristic, not semantic verification -- a coincidental number, a right-answer-for-the-
wrong-reason, or an oddly worded sentence can all fool it. **Layer 8** replaces this pattern-match
with an LLM actually reading the claim against the trace -- strictly more capable, but not
infallible either: a confidently-wrong reviewer is just as possible as a confidently-wrong
regex. Neither one substitutes for a human reading the actual evidence when the stakes are real;
matching the verifier to the claim (and knowing exactly what that verifier can't catch) is the
point of this whole section.

In [ ]:
show_layer_diagram(**DIAGRAMS["d6_evals"])

In [ ]:
import re as _re

def evaluate_tool_contracts(registry: ToolRegistry):
    failures = []
    for schema in registry.schemas():
        for field in ("name", "description", "parameters", "effect"):
            if not schema.get(field):
                failures.append(f"{schema.get('name', '<unnamed>')}: missing {field}")
    return {"eval": "tool_contracts", "passed": not failures, "failures": failures}

def evaluate_weather_response_shape(location: str = "Singapore"):
    reading = get_weather(location)
    required_numeric = ("temperature_c", "windspeed_kmh")
    missing = [k for k in required_numeric if not isinstance(reading.get(k), (int, float))]
    return {"eval": "weather_response_shape", "passed": not missing, "missing": missing, "sample": reading}

def check_answer_matches_tool_data(answer_text: str, tool_reading: dict, tolerance_c: float = 1.0):
    """End-to-end check: does the model's own prose state the temperature the tool actually
    returned? A component check (schema valid, fields present) says nothing about this -- the
    model can call the tool correctly and still misstate the number in its final answer.

    This is still a heuristic, not semantic verification: it looks for a number specifically in
    temperature context (adjacent to C/degrees, or near the word "temperature") rather than any
    digit in the text -- a prose answer that also mentions windspeed or a timestamp has other
    numbers in it, and a naive "any number matches" check can pass on a coincidental match to one
    of those instead of the model's actual temperature claim. See the discussion above (and
    Layer 8's fact-check reviewer) for why even this improved version is not the final word."""
    pattern = r'(-?\d+(?:\.\d+)?)\s*°?\s*C\b|temperature[^\d\-]{0,20}(-?\d+(?:\.\d+)?)'
    candidates = [float(m.group(1) or m.group(2)) for m in _re.finditer(pattern, answer_text, flags=_re.IGNORECASE)]
    true_temp = tool_reading["temperature_c"]
    match = any(abs(n - true_temp) <= tolerance_c for n in candidates)
    return {"eval": "answer_matches_tool_data", "passed": match,
            "true_temperature_c": true_temp, "temperature_numbers_found": candidates}

component_results = [
    evaluate_tool_contracts(full_registry),
    evaluate_weather_response_shape("Singapore"),
]
print("COMPONENT EVALS")
print(json.dumps(component_results, indent=2))

print("\nEND-TO-END CHECK on the Layer 5 grounded revision")
print(json.dumps(check_answer_matches_tool_data(grounded_revision.text, real_reading), indent=2))
print("\n(This still passes on a heuristic match, not proof of correctness -- compare with Layer 8's reviewer.)")

### Checkpoint D -- turn a failure into a regression

Write one regression specification:

| Inputs / preconditions | Action sequence | Expected observation | Layer | Claim supported | Remaining limit |
| --- | --- | --- | --- | --- | --- |
| | | | | | |

"Add more tests" is insufficient. State the state you create and the observation that decides the claim.

## 9. Layer 7 -- Planning without arbitrary code execution

The reference course shows planning in generated Python. Here the model produces an allowlisted
JSON plan. The host validates every tool name and argument before executing any step.

This retains the important idea -- model-generated action sequence -- without calling `exec()`
on arbitrary output.

In [ ]:
show_layer_diagram(**DIAGRAMS["d7_planning"])

In [ ]:
plan_reply = backend.chat([
    {"role": "system", "content": "You plan only with the supplied tools."},
    {"role": "user", "content": f'''
RETURN A PLAN JSON with keys plan and rationale.
Each plan item must be an object: {{"tool": "<name>", "args": {{}}}}. A bare tool name is invalid.
Goal: look up the real current weather in Berlin, then remember Berlin as the session default.
Available tools:
{json.dumps(full_registry.schemas(), indent=2)}
'''},
])

try:
    plan_object = extract_json_object(plan_reply.text)
except Exception as exc:
    plan_object = {"plan": [], "parse_error": str(exc), "raw": plan_reply.text}

def validate_plan(plan_object, registry):
    errors = []
    plan = plan_object.get("plan", [])
    if not isinstance(plan, list) or not plan:
        errors.append("plan must be a non-empty list")
        return errors
    for i, step in enumerate(plan):
        # A small model may return a bare tool-name string instead of {"tool":..., "args":...}.
        # Record that as a validation error rather than crashing on step.get(...).
        if not isinstance(step, dict):
            errors.append(
                f"step {i}: not an object (got {type(step).__name__}: {step!r}); "
                'expected {"tool": ..., "args": {...}}'
            )
            continue
        if step.get("tool") not in registry.tools:
            errors.append(f"step {i}: disallowed tool {step.get('tool')}")
        if not isinstance(step.get("args", {}), dict):
            errors.append(f"step {i}: args must be an object")
    return errors

plan_errors = validate_plan(plan_object, full_registry)
print("MODEL:", plan_reply.model)
print(json.dumps(plan_object, indent=2))
print("VALIDATION:", plan_errors or "PASS")

plan_execution = []
if not plan_errors:
    session.reset()
    for i, step in enumerate(plan_object["plan"]):
        try:
            result = full_registry.execute(step["tool"], step.get("args", {}))
            plan_execution.append({"step": i, "tool": step["tool"], "observation": result})
        except Exception as exc:
            plan_execution.append({"step": i, "tool": step.get("tool"), "error": str(exc)})
            break
print("\nEXECUTION TRACE")
print(json.dumps(plan_execution, indent=2, default=str))

## 10. Layer 8 -- Planner and Fact-Check Reviewer

A second role can improve separation of concerns, but it is not automatically independent
evidence. The reviewer receives:

- the model and role that produced the plan;
- the validated execution trace (the real tool data);
- a claimed final answer; and
- explicit trace identifiers.

Audit whether the reviewer stays within those inputs, and whether it actually catches a
number that doesn't match the retrieved data.

In [ ]:
show_layer_diagram(**DIAGRAMS["d8_multiagent"])

In [ ]:
claimed_answer = backend.chat([
    {"role": "system", "content": "Summarize the plan execution as a short final answer to the user."},
    {"role": "user", "content": f"Execution trace:\n{json.dumps(plan_execution, indent=2, default=str)}\n\nWrite the final answer."},
])

handoff = {
    "artifact": "weather-plan",
    "version": "v3-live-run",
    "planner_model": plan_reply.model,
    "trace_ids": [f"plan-{i:02d}" for i in range(len(plan_execution))],
    "execution": plan_execution,
    "claimed_answer": claimed_answer.text,
    "scope": "real Open-Meteo API data + session memory only",
}

reviewer_reply = backend.chat([
    {"role": "system", "content": "You are a fact-check reviewer. Use only the supplied evidence and state what remains unverified."},
    {"role": "user", "content": (
        "FACT-CHECK REVIEWER: does the claimed_answer's stated numbers match the execution trace's "
        "tool observations? Issue faithful / partially faithful / unfaithful, and name any mismatch.\n"
        + json.dumps(handoff, indent=2, default=str)
    )},
])

print("CLAIMED ANSWER MODEL:", claimed_answer.model)
print(claimed_answer.text)
print("\nREVIEWER MODEL:", reviewer_reply.model)
print(reviewer_reply.text)
print("\nHANDOFF RECORD")
print(json.dumps(handoff, indent=2, default=str))

In [ ]:
run_bundle = {
    "backend": "openrouter" if RUN_LIVE else "scripted-validation",
    "base_model": direct_reply.model,
    "direct_answer": direct_reply.text,
    "restricted_trace_remember": restricted_run_remember.trace,
    "restricted_trace_recall": restricted_run_recall.trace,
    "full_trace_remember": full_run_remember.trace,
    "full_trace_recall": full_run_recall.trace,
    "reflection_model": reflection_only.model,
    "reflection_only": reflection_only.text,
    "reflection_only_revision_model": reflection_only_revision.model,
    "reflection_only_revision": reflection_only_revision.text,
    "grounded_revision_model": grounded_revision.model,
    "grounded_revision": grounded_revision.text,
    "real_reading": real_reading,
    "component_evals": component_results,
    "plan_model": plan_reply.model,
    "plan": plan_object,
    "plan_validation": plan_errors,
    "plan_execution": plan_execution,
    "claimed_answer_model": claimed_answer.model,
    "claimed_answer": claimed_answer.text,
    "reviewer_model": reviewer_reply.model,
    "review": reviewer_reply.text,
    "handoff": handoff,
}

with open("week09_agent_colab-v3_run.json", "w", encoding="utf-8") as f:
    json.dump(run_bundle, f, indent=2, default=str)

print("Saved week09_agent_colab-v3_run.json")

## 11. Mini-assignment deliverable

**Submit one PDF report** (export a copy of `week09_agent_colab-v3_run.json` alongside it -- do not
submit only the JSON, and do not submit a raw `.ipynb`).

For **every** lettered section below, include at least one screenshot or pasted image of the actual
notebook cell input and its output that you are citing. A quoted trace ID with no accompanying
screenshot earns no evidence credit for that row. Screenshots must show your own run (visible
model identifier, timestamps, or the real API's `observed_at` field); a screenshot copied from a
classmate or from this brief is not evidence of your run.

### Agent failure-mode taxonomy

Classify every finding in sections B, C, and E as one of:

- **Capability gap, honest** -- the agent correctly declines or omits a task it cannot do.
- **Capability gap, overclaim** -- the agent implies success on a capability it does not have.
- **Silent state leak** -- a tool's side effect exposes information across a capability boundary
  that should not be visible (e.g. a "read-only" step surfaces state that only a state-changing
  tool should have been able to produce).
- **Malformed reply** -- the model's raw output is not valid JSON and gets caught as a recorded
  error, not a crash.
- **Template/placeholder echo** -- the model copies instruction text verbatim instead of writing a
  real answer.
- **Reasoning leakage** -- chain-of-thought text leaks into what should be a clean final answer.
- **Unverified claim** -- the final answer states something the trace's own evidence does not
  actually support.

A finding that does not fit any category is still valid evidence -- name it and explain why the
taxonomy misses it.

### A. Progressive component map -- 10 points

For each layer -- base LLM, agent policy, environment (`ChatSession`), tools, runtime, memory/trace,
evaluator, planner/reviewer, human gate -- name the notebook object or cell that implements it and
one failure it can introduce. Include a screenshot of the cell(s) you are citing for at least three
layers.

### B. Missing-capability probe -- 20 points

Run section 6.1's two conversations yourself (Conversation A: the "remember" request; Conversation
B: the later "my current city" query that depends on it). Cite at least three trace IDs across both
conversations, each with a screenshot. Classify Conversation B's behavior against the taxonomy
above, and explain what the missing "remember" capability caused to go wrong -- or be honestly
admitted.

### C. Full-capability probe -- 15 points

Run section 6.2's two conversations. Contrast Conversation B's outcome here against section B's
Conversation B: same-looking question, different evidence backing. Show the model request, host
validation/execution, the **real** returned observation, and the final claim it supports -- or
fails to support -- each backed by a screenshot.

### D. Reflection and evaluation -- 15 points

Compare reflection-only and evidence-grounded revision (section 7), with a screenshot of each FINAL
OUTPUT and of the real API reading. Include one component-eval result and the end-to-end
`answer_matches_tool_data` result (section 8). Explain why neither a critic nor a passing component
check establishes that the final prose is faithful to the retrieved data.

### E. Your own probe -- 20 points

Change one condition **not shown in sections B/C/D** and run it yourself -- for example, a
different city, a deliberately ambiguous or malformed request, or a task the tool registry cannot
support at all. Report only what your own run actually produced. Cite the trace IDs, classify the
outcome against the taxonomy above, and explain what it tells you that sections B-D did not already
show.

### F. Planning, handoff, and regression -- 10 points

Audit the plan validation and the fact-check reviewer's verdict (sections 9-10), with screenshots
of the plan JSON, the validation output, the claimed answer, and the reviewer's reply. Propose one
regression scenario with input state, expected observation, layer, and remaining limit.

### G. Acceptance and disclosure -- 10 points

Issue accept / accept-with-revisions / reject for the **chatbot's answer to the user** and
separately for **trusting this agent to act unsupervised on your behalf** (e.g. booking something
based on its weather read). Record the model identifier actually returned, run variability, any
empty/malformed replies, AI assistance, manual checks, and one generated suggestion you rejected or
bounded. Screenshot the `run_bundle` summary (`config` print and the final export cell's output) as
supporting evidence.

**Variability rule:** grading is based on trace-grounded interpretation, not whether the model chose
an instructor-preferred sequence. A malformed JSON action, an empty/`null` model reply, an invalid
tool request, a premature final answer, or a rate-limit event is valid evidence if documented,
screenshotted, and analysed.

### H. Feedback on this mini-assignment -- required, not scored on content

This section does not affect your grade on correctness, but it must be completed for the submission
to count as finished. In a few sentences each:

- What was confusing, underspecified, or took longer than it should have?
- Where did the notebook's behaviour (an empty model reply, a rate limit, a strange tool result) get
  in the way of demonstrating what you understood, rather than testing your understanding?
- One concrete change you would make to this notebook or assignment brief, and why.

Be specific -- "it was fine" or "make it easier" earns no useful signal. Cite a cell ID, section
letter, or trace ID wherever you can.

## 12. Takeaways and source notes

- A base LLM generates text; an agent adds an environment, actions, real observations, state, and
  stopping rules.
- Tools are capability and permission boundaries, not decorations around a prompt -- and here, the
  observations are genuinely real API data, not a simulation.
- Reflection revises a candidate's wording; a real external reading changes what is known.
- Component evals diagnose the pieces; a scenario/end-to-end eval is the only thing that checks
  whether the model's own prose is faithful to what a tool actually returned.
- Planning and multiple agents expand the action and verification burden.
- The final acceptance decision remains a human engineering responsibility.

Adapted conceptually from the DeepLearning.AI Agentic AI lab progression under
`references/week08_slides/notebooks/`: reflection, tools (including the weather-tool pattern in
`M3_UGL_1.ipynb`), component evals, planning, and multi-agent workflows. The runtime is original to
ESP3201; the weather tools call the real, free, keyless Open-Meteo API.

Backend documentation:

- OpenRouter `nvidia/nemotron-3-nano-30b-a3b:free`: https://openrouter.ai/nvidia/nemotron-3-nano-30b-a3b:free
- OpenRouter API quickstart: https://openrouter.ai/docs/quickstart
- Open-Meteo geocoding API: https://open-meteo.com/en/docs/geocoding-api
- Open-Meteo forecast API: https://open-meteo.com/en/docs